In [0]:
%run ./00_config_and_logging

In [0]:



#defining Gold table names
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)
from datetime import datetime
import uuid

GOLD_DIM_CVE = f"{CATALOG}.{SCHEMA}.gold_dim_cve"
GOLD_DIM_DATE = f"{CATALOG}.{SCHEMA}.gold_dim_date"
GOLD_DIM_SEVERITY = f"{CATALOG}.{SCHEMA}.gold_dim_severity"

print("Gold dimensions notebook started.")

In [0]:

#Reusable idempotent MERGE function
def merge_dimension(source_df, target_table, key_condition, view_name):
    """
    Creates the table if it does not exist, then runs an idempotent MERGE.
    The record_hash prevents unnecessary updates when no values changed.
    """

    # First execution only: create empty Delta table.
    source_df.limit(0).write.format("delta") \
        .mode("ignore") \
        .saveAsTable(target_table)

    source_df.createOrReplaceTempView(view_name)

    spark.sql(f"""
        MERGE INTO {target_table} t
        USING {view_name} s
        ON {key_condition}

        WHEN MATCHED AND NOT (t.record_hash <=> s.record_hash)
            THEN UPDATE SET *

        WHEN NOT MATCHED
            THEN INSERT *
    """)

    metrics = (
        spark.sql(f"DESCRIBE HISTORY {target_table}")
        .filter("operation = 'MERGE'")
        .orderBy(F.col("version").desc())
        .first()["operationMetrics"]
    )

    inserted = int(metrics.get("numTargetRowsInserted", 0))
    updated = int(metrics.get("numTargetRowsUpdated", 0))

    return inserted, updated


In [0]:
# Reading existing Silver unified CVE table


silver_cve = (
    spark.table(SILVER_UNIFIED)
    .filter(F.col("cve_id").isNotNull())
    .filter(F.trim(F.col("cve_id")) != "")
)

print(f"Silver CVE rows available: {silver_cve.count()}")

display(
    silver_cve.select(
        "cve_id",
        "description",
        "cvss_base_score",
        "cvss_severity",
        "published_date",
        "last_modified",
        "cwe_ids",
        "vuln_status"
    ).limit(10)
)


In [0]:
#Building the Severity dimension

severity_data = [
    (0, "UNKNOWN", 0, None, None),
    (1, "LOW", 1, 0.1, 3.9),
    (2, "MEDIUM", 2, 4.0, 6.9),
    (3, "HIGH", 3, 7.0, 8.9),
    (4, "CRITICAL", 4, 9.0, 10.0)
]

severity_schema = StructType([
    StructField("severity_key", IntegerType(), False),
    StructField("severity_name", StringType(), False),
    StructField("severity_rank", IntegerType(), False),
    StructField("cvss_score_min", DoubleType(), True),
    StructField("cvss_score_max", DoubleType(), True)
])

dim_severity = (
    spark.createDataFrame(severity_data, severity_schema)
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("severity_key").cast("string"),
                F.col("severity_name"),
                F.col("severity_rank").cast("string"),
                F.coalesce(F.col("cvss_score_min").cast("string"), F.lit("")),
                F.coalesce(F.col("cvss_score_max").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

display(dim_severity)


In [0]:
#Building the Date dimension

# Get the date range from the vulnerability records.
date_bounds = (
    silver_cve
    .select(
        F.to_date("published_date").alias("published_date"),
        F.to_date("last_modified").alias("last_modified_date"),
        F.to_date("kev_date_added").alias("kev_date_added_date"),
        F.to_date("kev_due_date").alias("kev_due_date_date")
    )
    .select(
        F.least(
            "published_date",
            "last_modified_date",
            "kev_date_added_date",
            "kev_due_date_date"
        ).alias("possible_min_date"),
        F.greatest(
            "published_date",
            "last_modified_date",
            "kev_date_added_date",
            "kev_due_date_date"
        ).alias("possible_max_date")
    )
    .agg(
        F.min("possible_min_date").alias("min_date"),
        F.max("possible_max_date").alias("max_date")
    )
)

display(date_bounds)

# Create a full calendar between the minimum and maximum vulnerability dates.
dim_date = (
    spark.range(1)
    .crossJoin(date_bounds)
    .select(
        F.explode(
            F.expr("sequence(min_date, max_date, interval 1 day)")
        ).alias("full_date")
    )
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter_number", F.quarter("full_date"))
    .withColumn("year_number", F.year("full_date"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("date_key").cast("string"),
                F.col("full_date").cast("string"),
                F.col("day_name"),
                F.col("month_name"),
                F.col("quarter_number").cast("string"),
                F.col("year_number").cast("string")
            ),
            256
        )
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

print(f"Date dimension rows: {dim_date.count()}")
display(dim_date.orderBy("full_date").limit(10))


In [0]:


# Building the CVE dimension


dim_cve = (
    silver_cve
    .select(
        F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"),
        F.col("cve_id"),
        F.col("description"),
        F.col("cwe_ids"),
        F.col("source_identifier"),
        F.col("vuln_status"),
        F.col("cvss_vector")
    )
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cve_id"), F.lit("")),
                F.coalesce(F.col("description"), F.lit("")),
                F.coalesce(F.concat_ws(",", F.col("cwe_ids")), F.lit("")),
                F.coalesce(F.col("source_identifier"), F.lit("")),
                F.coalesce(F.col("vuln_status"), F.lit("")),
                F.coalesce(F.col("cvss_vector"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

print(f"CVE dimension rows: {dim_cve.count()}")

display(
    dim_cve.select(
        "cve_key",
        "cve_id",
        "description",
        "cwe_ids",
        "vuln_status"
    ).limit(20)
)


In [0]:
#Merging the three dimensions and creating an audit log


run_id = new_run_id()
start = datetime.now()

total_inserted = 0
total_updated = 0

try:
    # Gold severity dimension
    inserted, updated = merge_dimension(
        dim_severity,
        GOLD_DIM_SEVERITY,
        "t.severity_key = s.severity_key",
        f"severity_dimension_updates_{uuid.uuid4().hex}"
    )

    total_inserted += inserted
    total_updated += updated

    print(f"gold_dim_severity | inserted={inserted}, updated={updated}")

    # Gold date dimension
    inserted, updated = merge_dimension(
        dim_date,
        GOLD_DIM_DATE,
        "t.date_key = s.date_key",
        f"date_dimension_updates_{uuid.uuid4().hex}"
    )

    total_inserted += inserted
    total_updated += updated

    print(f"gold_dim_date | inserted={inserted}, updated={updated}")

    # Gold CVE dimension
    inserted, updated = merge_dimension(
        dim_cve,
        GOLD_DIM_CVE,
        "t.cve_key = s.cve_key",
        f"cve_dimension_updates_{uuid.uuid4().hex}"
    )

    total_inserted += inserted
    total_updated += updated

    print(f"gold_dim_cve | inserted={inserted}, updated={updated}")

    # Pipeline audit log
    write_log(
        run_id,
        "Silver-to-Gold",
        "merge",
        SILVER_UNIFIED,
        "gold_dim_cve, gold_dim_date, gold_dim_severity",
        start,
        "Success",
        rows_inserted=total_inserted,
        rows_updated=total_updated
    )

    print(
        f"\nGold dimensions completed successfully. "
        f"Inserted: {total_inserted} | Updated: {total_updated}"
    )

except Exception as e:
    write_log(
        run_id,
        "Silver-to-Gold",
        "merge",
        SILVER_UNIFIED,
        "gold_dim_cve, gold_dim_date, gold_dim_severity",
        start,
        "Failure",
        error_message=str(e)
    )
    raise
